[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/gan.ipynb)

# Generative Adversarial Networks (GAN)

Are based on a strategy where two different deep networks are pitted against one another, with the goal of getting **one network to create new samples** that are not from the training data, but are so much like the training data that the other **network can’t tell the difference**.

* The **generator** transforms random noise into samples.
* The **discriminator** receives real samples and samples created by the generator, and outputs the probability of each one being real.

Both are trained at the same time: the discriminator learns to tell real from fake samples, and the generator learns to fool the discriminator.

In this example, let's train a generator to produce 2D points following a gaussian distribution (centred at (5, 5), with standard deviation 1), based on random noise.

## Import libraries

In [ ]:
import matplotlib.pyplot as plt
#importing Libraries
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## Understanding data

The real data are 2D points sampled from a gaussian distribution, with mean (5, 5) and variance 1 in each direction. The red circle has a radius of one standard deviation.

In [ ]:
from sklearn.datasets import make_blobs
from scipy.stats import multivariate_normal

X1, Y1 = make_blobs(n_samples=500, centers=[(5, 5)], cluster_std=1.0, random_state=0)

fig = plt.figure(figsize=(18, 8))

#Parameters to set
mu_x = 5
variance_x = 1.0

mu_y = 5
variance_y = 1.0

#Create grid and multivariate normal
x = np.linspace(2, 8, 30)
y = np.linspace(2, 8, 30)
X, Y = np.meshgrid(x, y)
pos = np.empty(X.shape + (2,))
pos[:, :, 0] = X;
pos[:, :, 1] = Y
rv = multivariate_normal([mu_x, mu_y], [[variance_x, 0], [0, variance_y]])

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(X, Y, rv.pdf(pos), rstride=1, cstride=1, linewidth=1, antialiased=False, cmap='viridis')
ax2 = fig.add_subplot(1, 2, 2)

ax2.scatter(X1[:, 0], X1[:, 1], marker="o", c=Y1, s=25, edgecolor="k")
cc = plt.Circle((5, 5), 1, fill=False, edgecolor='red', linewidth=2)
ax2.set_aspect(1)
ax2.add_artist(cc)

plt.show()

In [ ]:
print(X1.shape)
print(Y1.shape)

In [ ]:
def generate_norm_data(batch_size: int = 16):
    X1, Y1 = make_blobs(n_samples=batch_size, centers=[(5, 5)], cluster_std=1.0)
    return X1

In [ ]:
from sklearn.datasets import make_blobs

X1 = generate_norm_data(1000)

plt.figure(figsize=(8, 8))
ax2 = plt.gca()

plt.title("Samples from the target distribution", fontsize="small")
ax2.scatter(X1[:, 0], X1[:, 1], marker="o", s=25, edgecolor="k")
cc = plt.Circle((5, 5), 1, fill=False, edgecolor='red', linewidth=2)
ax2.set_aspect(1)
ax2.add_artist(cc)

plt.show()

In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


class NormDataset(Dataset):
    def __init__(self, n_samples=1000):
        self.Xs, self.y = make_blobs(n_samples=n_samples, centers=[(5, 5)], cluster_std=1.0)

    def __len__(self):
        return len(self.Xs)

    def __getitem__(self, idx):
        image = self.Xs[idx].astype(np.float32)
        label = self.y[idx]
        return image, label

## The generator

It receives a vector of 4 random values (noise) and outputs a 2D point.

In [ ]:
class Generator(nn.Module):

    def __init__(self):
        super().__init__()

        self.g = torch.nn.Sequential(
            torch.nn.Linear(4, 16),
            torch.nn.BatchNorm1d(16),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.g(x)

## The discriminator

It receives a 2D point and outputs the probability of it being real (sigmoid).

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.d = torch.nn.Sequential(
            torch.nn.Linear(2, 64),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(64, 64),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        res = self.d(x)
        return res


## Training

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
epochs = 1000
batch_size = 1000

training_data = NormDataset(n_samples=10000)
dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)

G = Generator()
D = Discriminator()
D.to(device)
G.to(device)

# Optimizers
G_optimizer = torch.optim.Adam(G.parameters(), lr=0.0002)
D_optimizer = torch.optim.Adam(D.parameters(), lr=0.0002)

loss = nn.BCELoss()

D_losses = []
G_losses = []

test_data = []
test_epochs = []

for epoch in range(epochs):
    for idx, (true_data, _) in enumerate(dataloader):
        # Training the discriminator
        # Real inputs are actual examples with gaussian distribution
        # Fake inputs are from the generator
        # Real inputs should be classified as 1 and fake as 0
        real_inputs = true_data.to(device)
        real_outputs = D(real_inputs)
        real_label = torch.ones(real_inputs.shape[0], 1).to(device)

        noise = torch.randn(real_inputs.shape[0], 4, device=device)
        # detach: the generator is not updated in this step
        fake_inputs = G(noise).detach()
        fake_outputs = D(fake_inputs)
        fake_label = torch.zeros(fake_inputs.shape[0], 1).to(device)

        outputs = torch.cat((real_outputs, fake_outputs), 0)
        targets = torch.cat((real_label, fake_label), 0)

        D_loss = loss(outputs, targets)
        D_optimizer.zero_grad()
        D_loss.backward()
        D_optimizer.step()

        # Training the generator
        # For generator, goal is to make the discriminator believe everything is 1
        noise = torch.randn(real_inputs.shape[0], 4, device=device)

        fake_inputs = G(noise)
        fake_outputs = D(fake_inputs)
        fake_targets = torch.ones([fake_inputs.shape[0], 1]).to(device)
        G_loss = loss(fake_outputs, fake_targets)
        G_optimizer.zero_grad()
        G_loss.backward()
        G_optimizer.step()

        G_losses.append(G_loss.item())
        D_losses.append(D_loss.item())

    if (epoch + 1) % 100 == 0:
        print('Epoch {}: discriminator_loss {:.3f} generator_loss {:.3f}'.format(epoch + 1, D_loss.item(), G_loss.item()))
        # keep some generated samples, to see the evolution of the generator
        G.eval()
        with torch.no_grad():
            noise = torch.randn(500, 4, device=device)
            test_data.append(G(noise).cpu().numpy())
        G.train()
        test_epochs.append(epoch + 1)

## Check training

The losses of the discriminator and of the generator, along the training. In a GAN, the losses don't simply decrease: the two networks compete, and the losses tend to stabilize when neither of them can improve against the other (the discriminator outputs 0.5 for every sample, and the loss is $\ln 2 \approx 0.693$).

In [ ]:
#plot the loss function
plt.plot(range(len(D_losses)), D_losses, label='Discriminator')
plt.plot(range(len(G_losses)), G_losses, label='Generator')
plt.axhline(np.log(2), color='gray', linestyle='--', label='ln 2')

plt.ylabel('Loss')
plt.xlabel('Batches')
plt.legend()
plt.show()

## Evolution of the GAN training

Samples created by the generator along the training. The red circle shows the target distribution (mean (5, 5), radius of one standard deviation).

In [ ]:
fig = plt.figure(figsize=(20, 8))

for i, generated_data in enumerate(test_data):
    plt.subplot(2, (len(test_data) + 1) // 2, i + 1)
    ax2 = plt.gca()

    plt.title("Epoch %d" % test_epochs[i], fontsize="small")
    ax2.scatter(generated_data[:, 0], generated_data[:, 1], marker="o", s=10, edgecolor="k")
    cc = plt.Circle((5, 5), 1, fill=False, edgecolor='red', linewidth=2)
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 10)
    ax2.set_aspect(1)
    ax2.add_artist(cc)

plt.show()

## Compare the generated and the real data

If the generator learned the distribution, the generated points should have a mean close to (5, 5) and a standard deviation close to 1 in each direction.

In [ ]:
G.eval()
with torch.no_grad():
    generated = G(torch.randn(10000, 4, device=device)).cpu().numpy()
real = generate_norm_data(10000)

print("Real data      - mean:", real.mean(axis=0).round(3), "std:", real.std(axis=0).round(3))
print("Generated data - mean:", generated.mean(axis=0).round(3), "std:", generated.std(axis=0).round(3))